In [ ]:
import pandas as pd
import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import os



import pyproj
# Point pyproj to the correct PROJ data directory
os.environ["PROJ_LIB"] = "<CONDA>/envs/EBMTest311/share/proj"
pyproj.datadir.set_data_dir(os.environ["PROJ_LIB"])
coastlines_file="<DATA_ROOT>/Raw/plate_model/StaticGeometries/Coastlines/Global_coastlines_low_res.shp"
coastlines_gdf=gpd.read_file(coastlines_file)

In [ ]:
deposits_gdf=gpd.read_file("<DATA_ROOT>/Raw/Vectors/Deposits/deposits_clipped.shp")
# deposits_gdf=deposits_gdf['']

In [ ]:
plt.hist(deposits_gdf['dev_status'])

In [ ]:
# Fill NaN tonnage values with 1 (as requested)
deposits_gdf['tonnage_for_plot'] = deposits_gdf['tonnage_mt']
# .fillna(1)
deposits_gdf=deposits_gdf.dropna(subset=['tonnage_for_plot'])
# Scale marker sizes: logarithmic transform for better visual spread
# Base size 80, scaled by log of tonnage
marker_sizes = 25 * np.log10(deposits_gdf['tonnage_for_plot'] + 1)

In [ ]:
# # ==============================================================================
# # STEP 1: LOAD AND PREPARE REAL MINERAL PROSPECTIVITY DATA
# # ==============================================================================
# print("\n" + "="*80)
# print("STEP 1: Load Real North American Copper Prospectivity Dataset")
# print("="*80)

# # Load the dataset
# # data_path = "<DATA_ROOT>/Raw/Vectors/Outputs/DatasetPU_USA/All_Data_Shuffled.csv"
# data_path= "<DATA_ROOT>/Raw/Vectors/STAMP_Training_data.csv"

# ALL_DATA = pd.read_csv(data_path)

# if data_path== "<DATA_ROOT>/Raw/Vectors/STAMP_Training_data.csv":
#     ALL_DATA['Longitude']=ALL_DATA['present_lon']
#     ALL_DATA['Latitude']=ALL_DATA['present_lat']

# print(f"Original dataset shape: {ALL_DATA.shape}")
# print(f"Total columns: {len(ALL_DATA.columns)}")

# # Remove unnecessary columns
# columns_to_remove = ['Unnamed: 0.2', 'Unnamed: 0', 'Unnamed: 0.1']
# ALL_DATA = ALL_DATA.drop(columns=[col for col in columns_to_remove if col in ALL_DATA.columns])

# print(f"\n✓ Removed {len([c for c in columns_to_remove if c in ALL_DATA.columns])} unnamed columns")
# print(f"Dataset shape after cleanup: {ALL_DATA.shape}")

# # Prepare weights from tonnage_mt
# # Handle -9999 as NaN
# ALL_DATA['tonnage_mt'] = ALL_DATA['tonnage_mt'].replace(-9999, np.nan)

# # Analyze tonnage distribution
# positive_mask = ALL_DATA['label_binary'] == 1
# positive_with_tonnage = positive_mask & ALL_DATA['tonnage_mt'].notna()

# if positive_with_tonnage.sum() > 0:
#     tonnage_values = ALL_DATA.loc[positive_with_tonnage, 'tonnage_mt']
    
#     print(f"\nTonnage distribution (deposits with known tonnage):")
#     print(f"  Count: {len(tonnage_values)}")
#     print(f"  Min: {tonnage_values.min():.2f} Mt")
#     print(f"  25th percentile: {tonnage_values.quantile(0.25):.2f} Mt")
#     print(f"  Median: {tonnage_values.median():.2f} Mt")
#     print(f"  75th percentile: {tonnage_values.quantile(0.75):.2f} Mt")
#     print(f"  Max: {tonnage_values.max():.2f} Mt")
#     print(f"  Mean: {tonnage_values.mean():.2f} Mt")
#     print(f"  Std: {tonnage_values.std():.2f} Mt")
    
#     # Plot tonnage distribution
#     fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
#     # Original scale
#     ax = axes[0]
#     ax.hist(tonnage_values, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
#     ax.set_xlabel('Tonnage (Mt)', fontsize=11)
#     ax.set_ylabel('Frequency', fontsize=11)
#     ax.set_title('Tonnage Distribution (Original Scale)', fontsize=12, fontweight='bold')
#     ax.grid(True, alpha=0.3)
    
#     # Log scale
#     ax = axes[1]
#     ax.hist(np.log1p(tonnage_values), bins=30, edgecolor='black', alpha=0.7, color='coral')
#     ax.set_xlabel('Log(1 + Tonnage)', fontsize=11)
#     ax.set_ylabel('Frequency', fontsize=11)
#     ax.set_title('Tonnage Distribution (Log Scale)', fontsize=12, fontweight='bold')
#     ax.grid(True, alpha=0.3)
    
#     plt.tight_layout()
#     plt.savefig('tonnage_distribution.png', dpi=300, bbox_inches='tight')
#     print(f"\n✓ Tonnage distribution plot saved to: tonnage_distribution.png")
#     plt.show()

# # Create weights: use log(1+tonnage) for positives to handle skewed distribution
# ALL_DATA['sample_weight'] = 1.0  # Default weight for unlabeled

# if positive_with_tonnage.sum() > 0:
#     # Use log transformation to reduce impact of extreme values - use directly
#     log_weights = ALL_DATA.loc[positive_with_tonnage, 'tonnage_mt']
#     ALL_DATA.loc[positive_with_tonnage, 'sample_weight'] = log_weights
    
#     print(f"\nWeight statistics (log-transformed, used directly):")
#     print(f"  Positive samples with tonnage: {positive_with_tonnage.sum()}")
#     print(f"  Positive samples without tonnage: {(positive_mask & ALL_DATA['tonnage_mt'].isna()).sum()}")
#     print(f"  Min weight (positive w/ tonnage): {log_weights.min():.3f}")
#     print(f"  Median weight (positive w/ tonnage): {log_weights.median():.3f}")
#     print(f"  Mean weight (positive w/ tonnage): {log_weights.mean():.3f}")
#     print(f"  Max weight (positive w/ tonnage): {log_weights.max():.3f}")
#     print(f"  Weight formula: log(1 + tonnage_mt)")
#     print(f"  Note: XGBoost/sklearn handle weights directly - no normalization needed")
# else:
#     print(f"\nNo tonnage data available - using uniform weights")


In [ ]:
# # Prepare deposit data with tonnage information
# deposits = ALL_DATA[ALL_DATA['label_binary'] == 1].copy()

# # Fill NaN tonnage values with 1 (as requested)
# deposits['tonnage_for_plot'] = deposits['tonnage_mt'].fillna(1)
    
# # Scale marker sizes: logarithmic transform for better visual spread
# # Base size 80, scaled by log of tonnage
# marker_sizes = 25 * np.log10(deposits['tonnage_for_plot'] + 1)

In [ ]:
# # columns_to_use2=['present_lon', 'present_lat', 'age (Ma)','tonnage_mt','subducted_carbonates_volume (m)', 'crustal_thickness_mean (m)']
# training_df_posc=deposits.copy()
# large_deposits = training_df_posc[training_df_posc['tonnage_mt'] >= 500]
# large_deposits_usa= large_deposits[(large_deposits['present_lon'] >= -125) & (large_deposits['present_lon'] <= -66) & (large_deposits['present_lat'] >= 24) & (large_deposits['present_lat'] <= 49)]
# large_deposits_usa=large_deposits_usa.sort_values('tonnage_mt')

In [ ]:
stamp_file="./Spatiotemporal/Max_STAMP.nc"
xr_spatial=xr.open_dataarray(stamp_file)
# xr_stamp

In [ ]:
spatial_file="./SpatialProspectivity/spatial_grid_predictions.nc"
xr_stamp=xr.open_dataarray(spatial_file)
# xr_spatial

In [ ]:
# First, rename xr_stamp dimensions to match xr_spatial
xr_stamp_renamed = xr_stamp.rename({'Latitude': 'Latitude', 'Longitude': 'Longitude'})

# Regrid xr_stamp to match xr_spatial's extent and resolution
xr_stamp_regridded = xr_stamp_renamed.interp(
    Latitude=xr_spatial.Latitude,
    Longitude=xr_spatial.Longitude,
    method='linear'
)

# Strategy 1: Multiplication (Ps × Pt) - emphasizes overlap of high values
combined_multiplicative = xr_spatial * xr_stamp_regridded
threshold = 0.00  # choose your cutoff

combined_multiplicative = combined_multiplicative.where(
    combined_multiplicative >= threshold
)
# Strategy 2: Maximum operator max(Ps, Pt) - highlights any strong signal
combined_maximum = xr.concat([xr_spatial, xr_stamp_regridded], dim='source').max(dim='source')

print("Original xr_stamp shape:", xr_stamp.shape)
print("Regridded xr_stamp shape:", xr_stamp_regridded.shape)
print("xr_spatial shape:", xr_spatial.shape)
print("\nCombined (multiplicative) range:", float(combined_multiplicative.min().values), "to", float(combined_multiplicative.max().values))
print("Combined (maximum) range:", float(combined_maximum.min().values), "to", float(combined_maximum.max().values))

In [ ]:
combined_multiplicative.name="combined_prospectivity"
combined_multiplicative.to_netcdf("Hyperdimensial_Prospectivity.nc")

In [ ]:
deposits=deposits_gdf.copy()


deposits['Longitude']=deposits['longitude']
deposits['Latitude']=deposits['latitude']
deposits=deposits[deposits['age_ma']<=170]


training_df_posc=deposits.copy()
large_deposits = training_df_posc[training_df_posc['tonnage_mt'] >= 500]
large_deposits_usa= large_deposits[(large_deposits['Longitude'] >= -125) & (large_deposits['Longitude'] <= -66) & (large_deposits['Latitude'] >= 24) & (large_deposits['Latitude'] <= 49)]
large_deposits_usa=large_deposits_usa.sort_values('tonnage_mt')
# Fill NaN tonnage values with 1 (as requested)
# deposits['tonnage_for_plot'] = deposits['tonnage_mt'].fillna(1)

# Scale marker sizes: logarithmic transform for better visual spread
# Base size 80, scaled by log of tonnage
marker_sizes = 25 * np.log10(deposits['tonnage_for_plot'] + 1)

In [ ]:
# Filter deposits to points that fall on valid (non-NaN) combined_multiplicative cells
sampled_vals = combined_multiplicative.interp(
    Longitude=xr.DataArray(deposits['Longitude'].values, dims='points'),
    Latitude=xr.DataArray(deposits['Latitude'].values, dims='points'),
    method='nearest'
)

valid_mask = sampled_vals.notnull().values
deposits_on_grid = deposits.loc[valid_mask].copy()
marker_sizes_on_grid = marker_sizes.loc[deposits_on_grid.index]


top = deposits_on_grid.nlargest(5, 'tonnage_for_plot')

print(f"Deposits total: {len(deposits)}")
print(f"Deposits on combined_multiplicative grid: {len(deposits_on_grid)}")
print(f"Dropped deposits: {len(deposits) - len(deposits_on_grid)}")

# Create figure and axis explicitly
fig, ax = plt.subplots(figsize=(8, 8), dpi=300)

cmap = plt.get_cmap("YlGnBu").copy()
cmap.set_bad(color='white')

im = combined_multiplicative.plot(
    ax=ax,
    cmap=cmap,
    robust=True,
    add_colorbar=False
)

cbar = plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("Hyperdimensional Prospectivity Score", fontsize=12)

# Plot only filtered deposits
ax.scatter(
    deposits_on_grid['Longitude'],
    deposits_on_grid['Latitude'],
    s=marker_sizes_on_grid,
    facecolor='none',
    edgecolors='red',
    linewidths=1.1,
    alpha=0.5,
    zorder=5
)
ax.scatter(
    top['Longitude'],
    top['Latitude'],
    s=marker_sizes.loc[top.index],
    color='#D95F02',
    # edgecolors='#D95F02',
    marker='*',
        linewidths=1.5,
        alpha=1.0,
        zorder=6 )
coastlines_gdf.plot(ax=ax, facecolor='None', linewidth=0.5,alpha=0.2, zorder=10)
ax.set_xlim(-145, -100)
ax.set_ylim(25, 72)
ax.set_xlabel("Longitude", fontsize=12)
ax.set_ylabel("Latitude", fontsize=12)
ax.grid(False)
ax.set_aspect('equal', adjustable='box')

for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1.2)

ax.set_title("Hyperdimensional Prospectivity Map", fontsize=14, fontweight='bold', pad=12)

plt.tight_layout()
plt.show()


In [ ]:
top = deposits_on_grid.nlargest(20, 'tonnage_for_plot')

In [ ]:
top


In [ ]:
# Filter deposits to points that fall on valid (non-NaN) combined_multiplicative cells
sampled_vals = combined_multiplicative.interp(
    Longitude=xr.DataArray(deposits['Longitude'].values, dims='points'),
    Latitude=xr.DataArray(deposits['Latitude'].values, dims='points'),
    method='nearest'
)

valid_mask = sampled_vals.notnull().values
deposits_on_grid = deposits.loc[valid_mask].copy()
marker_sizes_on_grid = marker_sizes.loc[deposits_on_grid.index]


top = deposits_on_grid.nlargest(20, 'tonnage_for_plot')
topc= top[8:9].copy()

print(f"Deposits total: {len(deposits)}")
print(f"Deposits on combined_multiplicative grid: {len(deposits_on_grid)}")
print(f"Dropped deposits: {len(deposits) - len(deposits_on_grid)}")

# Create figure and axis explicitly
fig, ax = plt.subplots(figsize=(8, 8), dpi=300)

cmap = plt.get_cmap("YlGnBu").copy()
cmap.set_bad(color='white')

im = combined_multiplicative.plot(
    ax=ax,
    cmap=cmap,
    robust=True,
    add_colorbar=False
)

cbar = plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("Hyperdimensional Prospectivity Score", fontsize=12)

# Plot only filtered deposits
ax.scatter(
    deposits_on_grid['Longitude'],
    deposits_on_grid['Latitude'],
    s=marker_sizes_on_grid,
    facecolor='none',
    edgecolors='red',
    linewidths=1.1,
    alpha=0.5,
    zorder=5
)
ax.scatter(
    topc['Longitude'],
    topc['Latitude'],
    # s=marker_sizes.loc[top.index],
    color='#D95F02',
    # edgecolors='#D95F02',
    marker='*',
        linewidths=1.5,
        alpha=1.0,
        zorder=6 )
coastlines_gdf.plot(ax=ax, facecolor='None', linewidth=0.5,alpha=0.2, zorder=10)
ax.set_xlim(-145, -100)
ax.set_ylim(25, 72)
ax.set_xlabel("Longitude", fontsize=12)
ax.set_ylabel("Latitude", fontsize=12)
ax.grid(False)
ax.set_aspect('equal', adjustable='box')

for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1.2)

ax.set_title("Hyperdimensional Prospectivity Map", fontsize=14, fontweight='bold', pad=12)

plt.tight_layout()
plt.show()


In [ ]:
# Filter deposits to points that fall on valid (non-NaN) combined_multiplicative cells
sampled_vals = combined_multiplicative.interp(
    Longitude=xr.DataArray(deposits['Longitude'].values, dims='points'),
    Latitude=xr.DataArray(deposits['Latitude'].values, dims='points'),
    method='nearest'
)

valid_mask = sampled_vals.notnull().values
deposits_on_grid = deposits.loc[valid_mask].copy()
marker_sizes_on_grid = marker_sizes.loc[deposits_on_grid.index]

print(f"Deposits total: {len(deposits)}")
print(f"Deposits on combined_multiplicative grid: {len(deposits_on_grid)}")
print(f"Dropped deposits: {len(deposits) - len(deposits_on_grid)}")

# Define percentile thresholds for discrete color bands
percentiles = [0, 50, 80, 90, 95, 98, 99, 100]  # 100%, 50%, 20%, 10%, 5%, 2%, 1%
thresholds = [combined_multiplicative.quantile(p/100).values for p in percentiles]

# Create labels with threshold values
label_names = ['50-100%', '20-50%', '10-20%', '5-10%', '2-5%', '1-2%', 'Top 1%']
labels = [f"{name}\n({thresholds[i]:.3f}-{thresholds[i+1]:.3f})" 
          for i, name in enumerate(label_names)]

# Create discrete color bands - Scientific publication color scheme
# Sequential from light (low prospectivity) to dark (high prospectivity)
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.pyplot as plt

# Professional color scheme: Light to dark, intuitive progression
# Using warm colors (yellow -> orange -> red -> dark red/purple)
colors = [
    "#f5f6d6",  # Very light yellow (lowest 50-100%)
    '#ffeda0',  # Light yellow-orange (20-50%)
    "#efa964",  # Yellow-orange (10-20%)
    "#c97240",  # Orange (5-10%)
    "#B24920",  # Dark orange (2-5%)
    "#622316",  # Red-orange (1-2%)
    "#000000",  # Dark red (Top 1% - highest)
]
cmap_discrete = ListedColormap(colors)
norm = BoundaryNorm(thresholds, cmap_discrete.N)

# Create figure and axis explicitly
fig, ax = plt.subplots(figsize=(10, 8), dpi=300)

# Plot with discrete colors
im = ax.pcolormesh(
    combined_multiplicative.Longitude,
    combined_multiplicative.Latitude,
    combined_multiplicative.values,
    cmap=cmap_discrete,
    norm=norm,
    shading='auto',
    zorder=1
)

# Add colorbar with custom labels including threshold values
cbar = plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("Hyperdimensional Prospectivity Percentile", fontsize=12, fontweight='bold')
cbar.set_ticks([(thresholds[i] + thresholds[i+1])/2 for i in range(len(thresholds)-1)])
cbar.set_ticklabels(labels, fontsize=9)

# Plot only filtered deposits
ax.scatter(
    deposits_on_grid['Longitude'],
    deposits_on_grid['Latitude'],
    s=marker_sizes_on_grid,
    facecolor='none',
    edgecolors='blue',
    linewidths=1.1,
    alpha=0.6,
    zorder=5,
    label='Known Deposits'
)

# Plot coastlines
coastlines_gdf.plot(ax=ax, facecolor='None', linewidth=0.5, alpha=0.3, zorder=10)

ax.set_xlim(-145, -100)
ax.set_ylim(25, 72)
ax.set_xlabel("Longitude", fontsize=12)
ax.set_ylabel("Latitude", fontsize=12)
ax.grid(False)
ax.set_aspect('equal', adjustable='box')

for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1.2)

ax.set_title("Hyperdimensional Prospectivity Map", 
             fontsize=14, fontweight='bold', pad=12)

# Add legend
ax.legend(loc='upper right', fontsize=10, framealpha=0.9)

plt.tight_layout()
plt.savefig('hyperdimensional_prospectivity_discrete_bands.png', dpi=300, bbox_inches='tight')
plt.show()

# Print threshold statistics
print("\n" + "="*60)
print("PERCENTILE THRESHOLD VALUES")
print("="*60)
for i, (label, thresh) in enumerate(zip(label_names, thresholds[:-1])):
    n_cells = ((combined_multiplicative >= thresh) & (combined_multiplicative < thresholds[i+1])).sum().values
    print(f"{label:12s} | Threshold: {thresh:.6f} - {thresholds[i+1]:.6f} | Cells: {n_cells:,}")
print("="*60)


In [ ]:
## hyperdimensional prospectivity distribution

combined_df=combined_multiplicative.to_dataframe().reset_index()
combined_df.hist("combined_prospectivity")

In [ ]:
# Create multi-panel plot showing top 1%, 2%, 5%, and 10% prospectivity AS POINTS
percentiles = [99, 98, 95, 90]  # Top 1%, 2%, 5%, 10%
labels = ['Top 1%', 'Top 2%', 'Top 5%', 'Top 10%']

# Calculate thresholds
thresholds = [combined_multiplicative.quantile(p/100).values for p in percentiles]

# Convert xarray to dataframe for point plotting
combined_df_plot = combined_multiplicative.to_dataframe('prospectivity').reset_index()
combined_df_plot = combined_df_plot.dropna(subset=['prospectivity'])

# Create 2x2 subplot
fig, axes = plt.subplots(2, 2, figsize=(16, 16), dpi=300)
axes = axes.flatten()

cmap = plt.get_cmap("YlGnBu")

for idx, (percentile, label, thresh) in enumerate(zip(percentiles, labels, thresholds)):
    ax = axes[idx]
    
    # Filter data to show only top percentile
    filtered_df = combined_df_plot[combined_df_plot['prospectivity'] >= thresh].copy()
    
    # Plot prospectivity as scatter points
    scatter = ax.scatter(
        filtered_df['Longitude'],
        filtered_df['Latitude'],
        c=filtered_df['prospectivity'],
        s=50,  # Point size
        cmap=cmap,
        alpha=1.0,
        vmin=thresh,
        vmax=combined_multiplicative.max().values,
        zorder=3
    )
    
    # Add colorbar
    cbar = plt.colorbar(scatter, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label("Prospectivity Score", fontsize=10)
    
    # Plot deposits
    ax.scatter(
        deposits_on_grid['Longitude'],
        deposits_on_grid['Latitude'],
        s=marker_sizes_on_grid,
        facecolor='none',
        edgecolors='red',
        linewidths=1.1,
        alpha=0.6,
        zorder=5
    )
    
    # Plot coastlines
    coastlines_gdf.plot(ax=ax, facecolor='None', edgecolor='black', linewidth=0.5, alpha=0.3, zorder=10)
    
    # Set extent and styling
    ax.set_xlim(-145, -100)
    ax.set_ylim(25, 72)
    ax.set_xlabel("Longitude", fontsize=11)
    ax.set_ylabel("Latitude", fontsize=11)
    ax.grid(False)
    ax.set_aspect('equal', adjustable='box')
    ax.set_facecolor('white')
    
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.2)
    
    # Title with threshold info
    ax.set_title(f"{label} (≥ {thresh:.4f})", fontsize=13, fontweight='bold', pad=10)
    
    # Count cells above threshold
    n_cells = len(filtered_df)
    total_cells = len(combined_df_plot)
    pct = (n_cells / total_cells) * 100
    
    # Add text box with statistics
    textstr = f'Points: {n_cells:,}\n({pct:.1f}% of grid)'
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
    ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=props)

plt.suptitle('Hyperdimensional Prospectivity: Multi-Threshold Analysis (Point Plot)', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('hyperdimensional_prospectivity_multi_threshold_points.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*70)
print("THRESHOLD STATISTICS")
print("="*70)
for label, thresh, percentile in zip(labels, thresholds, percentiles):
    filtered_df = combined_df_plot[combined_df_plot['prospectivity'] >= thresh]
    n_points = len(filtered_df)
    total_points = len(combined_df_plot)
    pct = (n_points / total_points) * 100
    print(f"{label:10s} | Threshold: {thresh:.6f} | Points: {n_points:,} ({pct:.2f}%)")
print("="*70)